In [1]:
import pandas as pd
import numpy as np
import random


for generating a dataset we first need to define the possible values of it

In [2]:
categories = [
    "Technology",
    "Sports",
    "Arts",
    "Business",
    "Science",
    "Community",
    "Health",
    "Medical",
    "Education"
]

tags_pool = {
    "Technology": [
        "AI", "Mobile Dev", "Web Dev", "Cybersecurity", "Data Science",
        "Programming", "Cloud"
    ],

    "Sports": [
        "Football", "Basketball", "Tennis", "Running", "Fitness"
    ],

    "Arts": [
        "Art", "Photography", "Music", "Design", "Writing"
    ],

    "Business": [
        "Business", "Entrepreneurship", "Finance", "Marketing", "Startup"
    ],

    "Science": [
        "Science", "Research", "Environment"
    ],

    "Community": [
        "Community"
    ],
    "Health": [
        "Health","nutrition","wellness","mental health","fitness","lifestyle"
    ],
    "Medical": [
        "Medical","medicine","healthcare","doctor","hospital","research"
    ],
    "Education": [
        "education","study","university","learning"
    ]
}



--------

now we will simulate users
300 rows

In [3]:
def generate_users(n=200):
    users = []
    all_tags = sum(tags_pool.values(), [])

    for _ in range(n):
        interests = random.sample(all_tags, k=random.randint(1, 3))
        users.append({"interests": interests})

    return users

users = generate_users(300)



In [17]:
def count_interested_users(event_tags, users):
    count = 0
    for user in users:
        if set(user["interests"]) & set(event_tags):
            count += 1
    return count


def simulate_attendance(interested_users, capacity, popularity, past_avg):

    if interested_users == 0:
        return 0


    base = interested_users * 0.7


    history_factor = past_avg * 0.3


    predicted = (base + history_factor) * popularity


    noise = random.uniform(-3, 3)

    predicted += noise


    return int(max(0, min(predicted, capacity)))



-----

now event dataset
500 rows

In [19]:

def generate_events(n=500):

    data = []

    category_popularity = {
        "Technology": 1.15,
        "Sports": 1.30,
        "Arts": 1.05,
        "Business": 1.00,
        "Science": 1.10,
        "Community": 0.95,
        "Health": 1.05,
        "Medical": 1.00,
        "Education": 1.10
    }

    past_attendance_by_category = {c: [] for c in categories}

    for i in range(n):

        category = random.choice(categories)

        available_tags = tags_pool.get(category, [])

        if len(available_tags) == 0:
            tags = []
        else:
            k = random.randint(1, min(4, len(available_tags)))
            tags = random.sample(available_tags, k=k)

        capacity = random.randint(50, 300)

        interested_users = count_interested_users(tags, users)

        # past average attendance
        past_list = past_attendance_by_category[category]

        if len(past_list) > 0:
            past_avg = np.mean(past_list)
        else:
            past_avg = random.randint(50, 100)

        attendance = simulate_attendance(
            interested_users,
            capacity,
            category_popularity[category],
            past_avg
        )

        past_attendance_by_category[category].append(attendance)

        data.append({
            "category": category,
            "capacity": capacity,
            "tags_count": len(tags),
            "interested_users_count": interested_users,
            "past_avg_attendance": past_avg,
            "attendance": attendance
        })

    return pd.DataFrame(data)


# Generate dataset

df = generate_events(500)


df.head()

,category,capacity,tags_count,interested_users_count,past_avg_attendance,attendance
0,Technology,86,4,59,84.0,78
1,Business,114,2,34,83.0,49
2,Sports,106,4,51,76.0,77
3,Science,241,2,26,100.0,52
4,Education,282,2,30,58.0,45


-----

save the dataset

In [20]:
df.to_csv("../data/events.csv", index=False)
